In [20]:
import pandas as pd 
import numpy as np 
import os 
import scipy.stats as stats
import category_encoders as ce
import matplotlib.pyplot as plt 
plt.rc("font", size=14)
import math 
# import seaborn as sns
# sns.set(style="white")
# sns.set(style="whitegrid", color_codes=True)
import string
from sklearn.compose import make_column_selector 
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import scale
import warnings
warnings.filterwarnings("ignore")
import datetime
from collections import Counter
from IPython.display import clear_output
from sklearn import datasets, linear_model
from sklearn.model_selection import LeaveOneOut
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score 
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import rand_score
from sklearn.metrics import adjusted_mutual_info_score
from sklearn.metrics import mutual_info_score
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error,  r2_score, explained_variance_score, make_scorer
from sklearn.model_selection import LeaveOneOut, cross_val_score,KFold
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import NearestNeighbors
%matplotlib inline

In [21]:
df=pd.read_excel("Real estate valuation data set.xlsx")
df

,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.916667,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.916667,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583333,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500000,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833333,5.0,390.56840,5,24.97937,121.54245,43.1
...,...,...,...,...,...,...,...,...
409,410,2013.000000,13.7,4082.01500,0,24.94155,121.50381,15.4
410,411,2012.666667,5.6,90.45606,9,24.97433,121.54310,50.0
411,412,2013.250000,18.8,390.96960,7,24.97923,121.53986,40.6
412,413,2013.000000,8.1,104.81010,5,24.96674,121.54067,52.5


## Outlier Handling

In [22]:
cat=[]
num=[]
for i in df.columns:
    if df[i].dtype=="object":
        cat.append(i)
    else:
        num.append(i)
print(cat) 
print(num) 

[]
['No', 'X1 transaction date', 'X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude', 'Y house price of unit area']


In [23]:
remove=['No',"Y house price of unit area"]
num=[x for x in num if x not in remove]
num

['X1 transaction date',
 'X2 house age',
 'X3 distance to the nearest MRT station',
 'X4 number of convenience stores',
 'X5 latitude',
 'X6 longitude']

In [24]:
import pickle

outlier_limits = {}

for col in num:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_limits[col] = (lower, upper)
   

In [25]:
outlier_limits

{'X1 transaction date': (2012.1666667, 2014.1666667),
 'X2 house age': (-19.6625, 56.8375),
 'X3 distance to the nearest MRT station': (-1458.1065000000003, 3201.7103),
 'X4 number of convenience stores': (-6.5, 13.5),
 'X5 latitude': (24.941317500000004, 24.999137499999996),
 'X6 longitude': (121.505255, 121.566135)}

In [26]:
pickle.dump(outlier_limits, open("outlier_limits.pkl", "wb"))
outlier_limits = pickle.load(open("outlier_limits.pkl", "rb"))


In [27]:
for col, (lower, upper) in outlier_limits.items():
    df[col] = df[col].clip(lower, upper)

In [28]:
from sklearn.impute import KNNImputer
knn_imputer = KNNImputer(n_neighbors=3)
df[num] = knn_imputer.fit_transform(df[num])
pickle.dump(knn_imputer, open("knn_imputer.pkl", "wb"))

In [29]:
df

,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.916667,32.0,84.87882,10.0,24.98298,121.540240,37.9
1,2,2012.916667,19.5,306.59470,9.0,24.98034,121.539510,42.2
2,3,2013.583333,13.3,561.98450,5.0,24.98746,121.543910,47.3
3,4,2013.500000,13.3,561.98450,5.0,24.98746,121.543910,54.8
4,5,2012.833333,5.0,390.56840,5.0,24.97937,121.542450,43.1
...,...,...,...,...,...,...,...,...
409,410,2013.000000,13.7,3201.71030,0.0,24.94155,121.505255,15.4
410,411,2012.666667,5.6,90.45606,9.0,24.97433,121.543100,50.0
411,412,2013.250000,18.8,390.96960,7.0,24.97923,121.539860,40.6
412,413,2013.000000,8.1,104.81010,5.0,24.96674,121.540670,52.5


In [30]:
## DiVide the dataset into indepent and dependent features
X=df.drop(["No", "Y house price of unit area"],axis=1)
y=df["Y house price of unit area"]

## Split the data in training and tetsing sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [31]:
from sklearn.preprocessing import StandardScaler
## Scale these features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [32]:
pickle.dump(scaler, open('scalar.pkl', 'wb'))

In [33]:
from sklearn.linear_model import LinearRegression
regr = LinearRegression()
regr.fit(X_train, y_train )
LR_y_pred = regr.predict(X_test) 

In [34]:
pickle.dump(regr, open('regr.pkl', 'wb'))

In [35]:
LR_y_pred

array([48.68299366, 41.40375603, 46.40681528, 42.32964266, 24.56977573,
       44.8131162 , 45.4813296 , 47.31816805, 23.33597463, 54.74716455,
       32.64664121, 34.14046879, 36.09114832, 22.95345902, 35.05398491,
       33.01174232, 43.0448133 , 45.98207237, 26.79232209, 44.04047051,
       10.29926831, 33.2264696 , 49.05782401, 45.95952174, 12.35690314,
       41.7724182 , 12.32673267, 44.96559173, 35.51262523, 37.92794347,
       14.8739904 , 39.03261475, 35.02385444, 26.75809   , 46.9581679 ,
       32.65776134, 51.10175191, 13.79846931, 48.32102246, 39.74936783,
       39.26141027, 41.13775501, 46.79859121, 38.24565674, 40.01906099,
       47.17974559, 43.20353179, 17.67009044, 48.17000542, 45.09187483,
       49.16340132, 49.86947525, 41.8121379 , 42.49338628, 36.76057619,
       14.73102095, 36.6252368 , 34.69594854, 25.44579829, 46.83776039,
       30.55533232, 31.19415373, 14.73102095,  9.96338659, 20.76531522,
       32.29391796, 27.65558967, 45.91335072, 33.73862036, 30.82